In [1]:
from flygym.compose import  KinematicPosePreset, ActuatorType, FlatGroundWorld
from flygym.assets.model.flybody.anatomy_flybody import (
    FlybodySkeleton,
    FlybodyJointPreset,
    FlybodyAxisOrder,
    FlybodyJointDOF,
    FlybodyBodySegment,
    FlybodyRotationAxis,
    FlybodyActuatedDOFPreset,
    FlybodyContactBodiesPreset
)

from flygym.utils.math import Rotation3D

from flygym.compose.fly import FlybodyFly
from flygym import Simulation


In [ ]:
axis_order = FlybodyAxisOrder.YAW_ROLL_PITCH
articulated_joints = FlybodyJointPreset.LEGS_ONLY
actuated_dofs = FlybodyActuatedDOFPreset.LEGS_ACTIVE_ONLY
actuator_type = ActuatorType.POSITION
neutral_pose = KinematicPosePreset.FLYBODY_NEUTRAL


# This controlls how strongly the actuators try to track the target joint angles
# actuator_gain = 150.0  # in uN*mm/rad (torque applied per angular discrepancy)
 
fly = FlybodyFly()
skeleton = FlybodySkeleton(axis_order=axis_order, joint_preset=articulated_joints)
fly.add_joints(skeleton, neutral_pose)
 
actuated_dofs_list = fly.skeleton.get_actuated_dofs_from_preset(actuated_dofs)
fly.add_actuators(
    actuated_dofs_list,
    actuator_type=actuator_type,
)

fly.add_tendons()
fly.add_tendon_actuators()

fly.add_leg_adhesion()
 
fly.colorize()

# tracking_cam = fly.add_tracking_camera(pos_offset=(0.5, -0.5, 0.2),
#                                        rotation=Rotation3D("xyaxes", (0.7, 0.7, 0, -0.2, 0.2, 0.95))
#                                        )

tracking_cam = fly.add_tracking_camera()

Applying wing default pose correction.


In [ ]:
bodysegs_with_ground_contact = FlybodyContactBodiesPreset.LEGS_THORAX_ABDOMEN_HEAD

spawn_position = (0, 0, 2.0)  # xyz in mm
spawn_rotation = Rotation3D("quat", (1, 0, 0, 0))  # wxyz in quaternion

world = FlatGroundWorld()
world.add_fly(
    fly,
    spawn_position,
    spawn_rotation,
    bodysegs_with_ground_contact=bodysegs_with_ground_contact,
    add_ground_contact_sensors=True,
)

sim = Simulation(world)
sim.set_renderer(tracking_cam)

In [4]:
from pathlib import Path
import pickle
import numpy as np

# load results
res_path = Path("/Users/stimpfli/Desktop/seqikpyv2/ik_results.pkl")
with open(res_path, "rb") as f:
    res = pickle.load(f)

recording_fps = 100
recording_timestep = 1 / recording_fps
mujoco_timestep = sim.mj_model.opt.timestep
# interpolate results to match mujoco timestep
interpolated_results = {}
for leg in res["legs"]:
    ik_results = res["ik_results"][leg][:, 1:]
    n_frames = ik_results.shape[0]
    recording_time = n_frames * recording_timestep
    mujoco_time = np.arange(0, recording_time, mujoco_timestep)
    interpolated_results[leg] = np.zeros((len(mujoco_time), ik_results.shape[1]))
    for j in range(ik_results.shape[1]):
        interpolated_results[leg][:, j] = np.interp(mujoco_time, np.arange(0, recording_time, recording_timestep), ik_results[:, j])

In [5]:
segment_mapping = {"ThC":["thorax", "coxa"],
                   "CTr":["coxa", "trochanterfemur"],
                   "FTi": ["trochanterfemur", "tibia"],
                   "TiTa":["tibia", "tarsus1"]}



translated_results = {}
for leg in res["legs"]:
    for i, joint_name in enumerate(res["joint_names"]):
        if "_" in joint_name:
            seg_name, dof_name = joint_name.split("_")
            parent_seg_name = segment_mapping[seg_name][0]
            child_seg_name = segment_mapping[seg_name][1]
            parent_name = f"c_{parent_seg_name}" if parent_seg_name == "thorax" else f"{leg.lower()}_{parent_seg_name}"
            child_name = f"{leg.lower()}_{child_seg_name}"
            joint = FlybodyJointDOF(
                parent=FlybodyBodySegment(parent_name),
                child=FlybodyBodySegment(child_name),
                axis=FlybodyRotationAxis(dof_name),
            )
            translated_results[joint] = interpolated_results[leg][:, i]
            
translated_results

{FlybodyJointDOF(parent=FlybodyBodySegment(name='c_thorax'), child=FlybodyBodySegment(name='rf_coxa'), axis=<FlybodyRotationAxis.YAW: 'yaw'>): array([ 0.02124628,  0.02215703,  0.02306777, ..., -0.34972242,
        -0.34972242, -0.34972242], shape=(10000,)),
 FlybodyJointDOF(parent=FlybodyBodySegment(name='c_thorax'), child=FlybodyBodySegment(name='rf_coxa'), axis=<FlybodyRotationAxis.ROLL: 'roll'>): array([0.8, 0.8, 0.8, ..., 0.8, 0.8, 0.8], shape=(10000,)),
 FlybodyJointDOF(parent=FlybodyBodySegment(name='c_thorax'), child=FlybodyBodySegment(name='rf_coxa'), axis=<FlybodyRotationAxis.PITCH: 'pitch'>): array([0.03868006, 0.03877301, 0.03886596, ..., 0.48295663, 0.48295663,
        0.48295663], shape=(10000,)),
 FlybodyJointDOF(parent=FlybodyBodySegment(name='rf_coxa'), child=FlybodyBodySegment(name='rf_trochanterfemur'), axis=<FlybodyRotationAxis.ROLL: 'roll'>): array([0.52270889, 0.5227021 , 0.52269531, ..., 0.06312453, 0.06312453,
        0.06312453], shape=(10000,)),
 FlybodyJointD

In [ ]:
from tqdm import trange

fly_name = fly.name

joint_angles_nmf = []
for joint, act in fly.jointdof_to_mjcfactuator_by_type[actuator_type].items():
    joint_angles_nmf.append(translated_results[joint])
joint_angles_nmf = np.array(joint_angles_nmf).T  # shape (nsteps, n_dofs)

nsteps_sim = joint_angles_nmf.shape[0]
n_dofs = len(fly.get_jointdofs_order())
n_actuated_dofs = len(fly.get_actuated_jointdofs_order(actuator_type))
 
simulated_joint_angles = np.full((nsteps_sim, n_dofs), np.nan, dtype=np.float32)
actuator_torques = np.full((nsteps_sim, n_actuated_dofs), np.nan, dtype=np.float32)
 
sim.reset()
# Turn adhesion on for all 6 legs
sim.set_leg_adhesion_states(fly_name, np.zeros(6, dtype=np.bool))
sim.set_tendon_actuator_inputs(fly_name, [0.2, 0.0, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5])
# sim.warmup()
 
for step_idx in trange(nsteps_sim, desc="Simulating"):
    # Set actuator inputs
    
    target_angles = joint_angles_nmf[step_idx, :]
    sim.set_actuator_inputs(fly_name, actuator_type, target_angles)
    
    # Step simulation
    sim.step_with_profile()  # can be replaced with sim.step()
    
    # Record simulation data
    simulated_joint_angles[step_idx, :] = sim.get_joint_angles(fly_name)
    actuator_torques[step_idx, :] = sim.get_actuator_forces(fly_name, actuator_type)
    
    # Render as needed (flygym internally decides whether to actually render a frame
    # based on renderer configs)
    sim.render_as_needed_with_profile()  # can be replaced with sim.render_as_needed()

Simulating: 100%|██████████| 10000/10000 [00:01<00:00, 5251.67it/s]


In [7]:
sim.renderer.show_in_notebook()

nmf/trackcam This browser does not support the video tag.
